# Challenge 8 · Teach a Quantum Circuit
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hastikacheddy/qiskit-fall-fest-participant/blob/main/challenges/challenge_08_quantum_ml/challenge_08_quantum_ml.ipynb)

**Tier 3 - Advanced Challenges** · **Estimated time: 60-90 minutes** (an estimate, not a guarantee) · *Participant notebook*

> Train a two-qubit variational circuit to classify points in the plane, evaluate it honestly on held-out data, and compare it with a classical baseline - without leaking test data into training.

### Learning objectives
- Encode classical features into a quantum circuit and read out a prediction.
- Train circuit parameters with a classical optimiser.
- Avoid train/test leakage and compute evaluation metrics correctly.
- Compare fairly with a classical baseline and discuss what the comparison does (not) show.

### Prerequisites
- Challenges 1-3 and 7 or equivalent
- Parameterised circuits
- Train/test splits and accuracy

### What you will write
| Function | Status |
|---|---|
| `scale_features` | required |
| `variational_ansatz` | required |
| `model_output` | required |
| `mse_loss` | required |
| `train_vqc` | required |
| `binary_metrics` | required |
| `run_experiment` | required |
| `parameter_shift_gradient` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `scale_features` fits min-max scaling to [0, pi] on the training data only (no train/test leakage).
- `variational_ansatz` returns a parameterised circuit with the documented dimensions.
- `model_output` is the exact <Z_0> of feature map plus ansatz: finite and within [-1, 1].
- `mse_loss` equals the mean squared error between model outputs and +-1 labels.
- `train_vqc` uses the documented seeded initialisation, is reproducible and never ends worse than it started.
- `binary_metrics` computes accuracy, precision, recall and F1 correctly, including degenerate cases.
- `run_experiment` is leakage-free: training never sees test data; reported test metrics are recomputed from the returned model.
- Invalid shapes, non-finite values, labels other than -1/+1, wrong parameter counts and invalid layer numbers raise ValueError; no function modifies its inputs.

**Quality benchmarks** (reported separately from correctness - a stochastic optimiser is not required to be perfect):
- Held-out accuracy is clearly above chance on the supplied dataset and the training loss falls (benchmark, not correctness).

**Bonus (optional, never required to pass):**
- `parameter_shift_gradient` matches finite differences.

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

# Colab opens a notebook straight from GitHub without the project next to it: the project is fetched from this repository.
PROJECT_REPOSITORY = 'https://github.com/hastikacheddy/qiskit-fall-fest-participant.git'

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    try:
        on_colab = importlib.util.find_spec('google.colab') is not None
    except ModuleNotFoundError:  # there is no `google` package at all
        on_colab = False
    if root is None and on_colab:
        # the notebook is here, the project is not: fetch it (a few seconds)
        root = here / PROJECT_REPOSITORY.rsplit('/', 1)[-1].removesuffix('.git')
        if not root.exists():
            print('Fetching the project from', PROJECT_REPOSITORY, '...')
            try:
                subprocess.check_call(['git', 'clone', '--quiet', '--depth', '1', PROJECT_REPOSITORY, str(root)])
            except (subprocess.CalledProcessError, OSError) as error:
                raise RuntimeError(f'Could not fetch the project from {PROJECT_REPOSITORY} ({error}). '
                                   "Upload the project yourself instead (see README.md, section 'Colab').") from None
    if root is None or not (root / 'pyproject.toml').is_file():
        raise RuntimeError('Open this notebook from inside the project folder (the one that contains pyproject.toml). '
                           'If you only have the notebook, download the whole project from ' + PROJECT_REPOSITORY.removesuffix('.git') +
                           " (Code > Download ZIP) or open the notebook in Google Colab - see README.md, section 'Colab'.")
    print('Installing the project and its libraries - about a minute the first time ...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
A sensor delivers **two measurements** per sample and a label **+1 or −1**. You will teach a **two-qubit variational circuit** to predict the label, **evaluate it honestly** on data it has never seen, and compare it with classical baselines - without letting test information leak into training.

1. `scale_features` - map the raw features to angles in [0, π], with statistics from the **training data only**.
2. `variational_ansatz` - the trainable part of the circuit.
3. `model_output` - the model: feature encoding + ansatz, read out as ⟨Z₀⟩.
4. `mse_loss` - mean squared error against the ±1 labels.
5. `train_vqc` - fit the circuit parameters with a classical optimiser (seeded, reproducible).
6. `binary_metrics` - accuracy, precision, recall, F1 (including the degenerate cases).
7. `run_experiment` - the whole pipeline, leakage-free.
8. *(Bonus)* `parameter_shift_gradient` - the exact gradient of the loss from circuit evaluations.

### Conventions (used by the notebook and by every test)
* **Data:** `X` has shape `(n, 2)`; labels `y` are `-1` or `+1` (labels 0 / 1 are invalid). The positive class is `+1`.
* **Scaling:** feature *j* ↦ `π · (x − min_j) / (max_j − min_j)` with `min_j`, `max_j` taken from the **training** data and applied unchanged to the test data (test values may leave [0, π]; no clipping). A feature that is constant in the training data maps to `π/2`.
* **The model** has two qubits that start in |00⟩ (qubit 0 is the rightmost bit):
  1. *feature encoding:* `RY(x₀)` on qubit 0 and `RY(x₁)` on qubit 1, with the **scaled** features as angles (radians);
  2. *ansatz with L layers:* layer *l* applies `RY(θ[4l])` on qubit 0, `RY(θ[4l+1])` on qubit 1, `RZ(θ[4l+2])` on qubit 0, `RZ(θ[4l+3])` on qubit 1, then `CX` with control qubit 0 and target qubit 1;
  3. *output:* f(x; θ) = ⟨Z₀⟩ ∈ [−1, 1] (exact, no sampling).
  The parameter vector θ has `4·L` entries in exactly this order.
* **Prediction:** `+1` if f ≥ 0, else `−1`. **Loss:** mean of (f(xᵢ) − yᵢ)².
* **Seeded start:** `train_vqc` begins at `rng = np.random.default_rng(seed)`, `θ₀ = rng.uniform(−π, π, 4·L)`.
* **Metrics:** TP, FP, TN, FN count the positive class `+1`; precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = 2PR/(P+R); a quotient with a zero denominator is `0.0`.

### Function contracts
The helpers `validate_features`, `validate_sample`, `validate_labels`, `validate_params`, `validate_num_layers` are **provided**: they raise `ValueError` for invalid input and return clean NumPy arrays.

| Function | Input | Output | Raises `ValueError` when |
|---|---|---|---|
| `scale_features(X_train, X_test)` | two 2-D arrays with the same number of columns | `(train_scaled, test_scaled)`, new float arrays | shapes / feature counts invalid, non-finite entries, no rows |
| `variational_ansatz(num_layers)` | integer L ≥ 1 | `QuantumCircuit` on 2 qubits with `4L` free parameters (a `ParameterVector`, so that `circuit.parameters` follows the documented order), no measurements | L is not an integer ≥ 1 |
| `model_output(x, params, num_layers)` | one scaled sample (length 2), `4L` parameters | `float` f(x; θ) | wrong length, non-finite values, invalid L |
| `mse_loss(X, y, params, num_layers)` | scaled samples `(n, 2)`, labels ±1, parameters | `float` mean squared error | shapes invalid, labels not ±1, invalid parameters |
| `train_vqc(X_train, y_train, num_layers=2, seed=0, maxiter=100)` | scaled training data, labels ±1 | `dict` with `params` (array of 4L), `initial_loss` (the loss at the seeded start), `final_loss` (= loss at `params`, never larger than `initial_loss`); an optional `history` list is ignored by the tests. Equal arguments → equal result | invalid data, `seed` not a non-negative integer, `maxiter` not a positive integer |
| `binary_metrics(y_true, y_pred)` | two label arrays of equal length | `dict` with `accuracy`, `precision`, `recall`, `f1` (floats) and `tp`, `fp`, `tn`, `fn` (ints) | empty / unequal length, labels not ±1 |
| `run_experiment(X_train, y_train, X_test, y_test, num_layers=2, seed=0, maxiter=100)` | **raw** train and test data | `dict`: `params`, `num_layers`, `feature_min`, `feature_max` (the **training** minima / maxima), `initial_loss`, `train_loss`, `test_loss` (MSE on the scaled data), `train_metrics`, `test_metrics` (as `binary_metrics`) | any invalid input |
| *(bonus)* `parameter_shift_gradient(X, y, params, num_layers)` | as `mse_loss` | NumPy array of the `4L` partial derivatives of the loss, from the parameter-shift rule | as `mse_loss` |

**Correctness versus quality.** The correctness tests never ask for an accuracy: they check scaling, circuit, model output, loss, metrics, reproducibility and the absence of leakage *exactly* (tolerance 1e-9). The **quality benchmark** is reported separately: with the defaults (`num_layers=2, seed=0, maxiter=100`) on the supplied dataset the held-out accuracy should be clearly above chance (≥ 70 %), and the training loss should fall by at least 30 %.

## Concept: a classifier made of rotations
* **Angle encoding.** A feature becomes the rotation angle of a qubit. One encoding layer means the model is a *trigonometric polynomial* of degree 1 in each feature: it can draw smooth, periodic boundaries - but not arbitrary ones.
* **Variational ansatz.** Trainable rotations and an entangling CX; a classical optimiser changes the parameters to lower the loss. The output ⟨Z₀⟩ is a number in [−1, 1], so ±1 labels and the squared error are a natural pair.
* **Honest evaluation.** The accuracy on the *training* data says how well the model memorised; only data that played **no role** in scaling, training or model choice says how well it generalises. Leakage - for example fitting the scaler on all data - makes a model look better than it is. Here it is small, in real projects it can be dramatic.
* **Fair comparison.** A baseline must see the same split and the same preprocessing rules. With ~60 test points an accuracy is uncertain by roughly ±10 percentage points (see the intervals), so "A beat B by 3 points" is not evidence.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from scipy.optimize import minimize
from qiskit_fest.challenges.c08_quantum_ml import (
    validate_features, validate_labels, validate_num_layers, validate_params, validate_sample)

In [ ]:
from qiskit_fest.challenges.c08_quantum_ml import (
    knn_baseline, logistic_baseline, make_dataset, plot_accuracy_comparison, plot_dataset, plot_decision_regions, plot_loss_history)
from qiskit_fest.viz import draw_circuit

X_train, y_train, X_test, y_test = make_dataset()       # 60 training and 60 test points, seed 3
print("training set:", X_train.shape, "labels +1 / -1:", int((y_train == 1).sum()), "/", int((y_train == -1).sum()))
print("raw feature ranges (training):", X_train.min(axis=0).round(2), "to", X_train.max(axis=0).round(2), " <- not angles in [0, pi]")
plot_dataset(X_train, y_train, X_test, y_test, "The data: two interleaving half circles with noise")

## Part 1 · Scale the features - without leaking  *(TODO)*
Complete `scale_features(X_train, X_test)`: map every feature of the **training** data to [0, π] and apply the **same** transformation to the test data.

<details><summary>💡 Hint 1 (subtle)</summary>

The scaling is defined by two numbers per feature: a minimum and a maximum. From *which* data may you compute them, if the test data is supposed to be unseen?
</details>

<details><summary>💡 Hint 2</summary>

`lo = train.min(axis=0)`, `hi = train.max(axis=0)`; both arrays get `(x - lo) / (hi - lo) * np.pi` with these same `lo`, `hi`. A test value outside the training range simply lands outside [0, π]: do not clip and do not refit. A constant training feature (`hi == lo`) would divide by zero - map it to π/2 instead. Use the provided `validate_features` first.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

```python
train = validate_features(X_train)
test = validate_features(X_test, ...)               # same number of features as the training array
lo, hi = ..., ...                                   # per-feature minimum and maximum: of which array?
# a feature with hi == lo has no spread: avoid dividing by zero and map it to np.pi / 2
# every other feature:  (data - lo) / (hi - lo) * np.pi   - for BOTH arrays, with the same lo and hi
return scaled_train, scaled_test
```
</details>

In [ ]:
def scale_features(X_train, X_test):
    """Min-max scale both arrays to angles with the minima / maxima of the TRAINING data (constant feature -> pi/2). Returns (train, test)."""
    # TODO: validate, take the statistics from X_train only, transform both arrays with them.
    raise NotImplementedError("TODO: implement scale_features()")

In [ ]:
with if_ready():
    scaled_train, scaled_test = scale_features(X_train, X_test)
    print("scaled training range:", scaled_train.min(axis=0).round(3), "...", scaled_train.max(axis=0).round(3), "(exactly 0 ... pi = 3.142)")
    print("scaled test range    :", scaled_test.min(axis=0).round(3), "...", scaled_test.max(axis=0).round(3), "(may leave [0, pi] - that is correct)")
    leaky_train, leaky_test = scale_features(np.vstack([X_train, X_test]), X_test)   # the WRONG way: statistics of all data
    print("largest change of a scaled test value if the test data were used for the scaling:", float(np.abs(leaky_test - scaled_test).max().round(3)),
          "radians - small here, but it is information the model must not have")

In [ ]:
report = run_public_checks(8, globals(), keyword="scale_features")

## Part 2 · The trainable circuit  *(TODO)*
Complete `variational_ansatz(num_layers)` as documented: L layers of `RY`, `RY`, `RZ`, `RZ` and a CX, with `4L` parameters in the documented order. Use a `ParameterVector` so that `circuit.parameters` is in index order.

<details><summary>💡 Hint 1 (subtle)</summary>

Count the free angles per layer: two `RY` and two `RZ`. Which Qiskit object holds a *list* of parameters and keeps their order?
</details>

<details><summary>💡 Hint 2</summary>

`theta = ParameterVector("theta", 4 * num_layers)`; inside the layer loop use `theta[4 * layer + k]` for k = 0 … 3 in the order RY(q0), RY(q1), RZ(q0), RZ(q1), then `qc.cx(0, 1)`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

```python
layers = validate_num_layers(num_layers)
theta = ParameterVector("theta", ...)               # how many angles in total?
qc = QuantumCircuit(2)
for l in range(layers):
    ...   # RY(theta[4l]) on q0, RY(theta[4l+1]) on q1, RZ(theta[4l+2]) on q0, RZ(theta[4l+3]) on q1, then ONE cx(0, 1)
return qc
```
</details>

In [ ]:
def variational_ansatz(num_layers: int) -> QuantumCircuit:
    """2-qubit circuit with 4 * num_layers free parameters: per layer RY(q0), RY(q1), RZ(q0), RZ(q1), CX(0, 1)."""
    # TODO: validate num_layers (ValueError), build the circuit with a ParameterVector.
    raise NotImplementedError("TODO: implement variational_ansatz()")

In [ ]:
with if_ready():
    ansatz = variational_ansatz(2)
    print("parameters:", ansatz.num_parameters, "(2 layers x 4)")
    draw_circuit(ansatz, "The ansatz with two layers")

In [ ]:
report = run_public_checks(8, globals(), keyword="variational_ansatz")

## Part 3 · The model  *(TODO)*
Complete `model_output(x, params, num_layers)`: encode the scaled sample with `RY(x₀)` on qubit 0 and `RY(x₁)` on qubit 1, append the ansatz with the given parameters and return the exact expectation value ⟨Z₀⟩.
Sanity check: with all parameters 0 and one layer the model is just ⟨Z₀⟩ = cos(x₀).

<details><summary>💡 Hint 1 (subtle)</summary>

Which Pauli string measures Z on qubit 0 of two qubits? Remember that Qiskit labels are written with the *rightmost* character on qubit 0.
</details>

<details><summary>💡 Hint 2</summary>

`ansatz.assign_parameters(params)` returns a circuit with numbers instead of parameters (the array is in `circuit.parameters` order). `Statevector(circuit).expectation_value(SparsePauliOp("IZ"))` gives the exact value; take the real part.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

```python
x = validate_sample(x)
params = validate_params(params, num_layers)
...   # a two-qubit circuit; encoding: one RY per feature, on qubit 0 and qubit 1
...   # append the ansatz with numbers bound to its parameters (qc.compose(...))
value = Statevector(qc).expectation_value(SparsePauliOp(...))     # which label measures Z on qubit 0?
return float(np.real(value))
```
</details>

In [ ]:
def model_output(x, params, num_layers: int) -> float:
    """f(x; theta) = <Z_0> after RY(x0) on qubit 0, RY(x1) on qubit 1 and the ansatz with the given parameters (exact)."""
    # TODO: validate, build encoding + ansatz, return the exact expectation value of Z on qubit 0.
    raise NotImplementedError("TODO: implement model_output()")

In [ ]:
theta0 = np.random.default_rng(0).uniform(-np.pi, np.pi, 8)      # random parameters for two layers (also used below)
with if_ready():
    scaled_train, scaled_test = scale_features(X_train, X_test)
    print("zero parameters, one layer:", round(model_output([0.7, 2.1], np.zeros(4), 1), 6), "  cos(0.7) =", round(float(np.cos(0.7)), 6))
    print("outputs of random parameters for three samples:", [round(model_output(s, theta0, 2), 3) for s in scaled_train[:3]])

In [ ]:
report = run_public_checks(8, globals(), keyword="model_output")

## Part 4 · The loss  *(TODO)*
Complete `mse_loss(X, y, params, num_layers)`: the mean over all samples of (f(xᵢ; θ) − yᵢ)², with labels **−1 / +1** (a label of 0 is invalid).

<details><summary>💡 Hint</summary>

Loop over the rows of `X` with your `model_output`, collect the outputs in an array and use `np.mean((outputs - y) ** 2)`. Validate with the provided helpers (`validate_features(X, 2)`, `validate_labels(y, len(X))`).
</details>

In [ ]:
def mse_loss(X, y, params, num_layers: int) -> float:
    """Mean squared error between the model outputs and the +-1 labels."""
    # TODO: validate, evaluate the model on every sample, return the mean squared error.
    raise NotImplementedError("TODO: implement mse_loss()")

In [ ]:
with if_ready():
    scaled_train, scaled_test = scale_features(X_train, X_test)
    print("loss of random parameters on the scaled training data:", round(mse_loss(scaled_train, y_train, theta0, 2), 4))
    print("a model that always answers 0 has loss 1; the worst possible loss is 4")

In [ ]:
report = run_public_checks(8, globals(), keyword="mse_loss")

## Part 5 · Train the circuit  *(TODO)*
Complete `train_vqc(X_train, y_train, num_layers=2, seed=0, maxiter=100)`: start at the documented seeded parameters, minimise the training loss with a classical optimiser (e.g. COBYLA with `maxiter`), and return `params`, `initial_loss` and `final_loss`. The result must be reproducible and **never worse than the starting point**.

<details><summary>💡 Hint 1 (subtle)</summary>

An optimiser can end somewhere worse than it began (or stop after a bad step). What should the function return if the final loss is larger than the initial one?
</details>

<details><summary>💡 Hint 2</summary>

`rng = np.random.default_rng(seed)`, `theta0 = rng.uniform(-np.pi, np.pi, 4 * num_layers)`. Objective `lambda p: mse_loss(X_train, y_train, p, num_layers)`; `scipy.optimize.minimize(objective, theta0, method="COBYLA", options={"maxiter": maxiter})` (COBYLA needs at least `4 * num_layers + 2` evaluations; with a smaller `maxiter` SciPy warns and raises it). Remember the loss at `theta0` as `initial_loss`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Record every objective value in a list called `history` (optional) - you can plot it afterwards. After the optimiser: `final = mse_loss(X_train, y_train, result.x, num_layers)`; if `final > initial`, return `theta0` and `initial` instead. The returned `final_loss` must be the loss *of the returned parameters*.
</details>

In [ ]:
def train_vqc(X_train, y_train, num_layers: int = 2, seed: int = 0, maxiter: int = 100) -> dict:
    """Train the circuit: dict with params, initial_loss (at the seeded start) and final_loss (never larger); reproducible."""
    # TODO: validate, seeded start theta0, minimise mse_loss, keep the better of start and result.
    raise NotImplementedError("TODO: implement train_vqc()")

In [ ]:
with if_ready():
    scaled_train, scaled_test = scale_features(X_train, X_test)
    fit = train_vqc(scaled_train, y_train, num_layers=2, seed=0, maxiter=60)
    print(f"training loss {fit['initial_loss']:.3f} -> {fit['final_loss']:.3f}")
    if "history" in fit:
        plot_loss_history(fit["history"])

In [ ]:
report = run_public_checks(8, globals(), keyword="train_vqc")

## Part 6 · Metrics you can trust  *(TODO)*
Complete `binary_metrics(y_true, y_pred)` for labels **−1 / +1** with `+1` as the positive class: counts `tp, fp, tn, fn` and `accuracy`, `precision`, `recall`, `f1`. A quotient with a zero denominator is `0.0` (for example precision when nothing is predicted positive).

<details><summary>💡 Hint 1 (subtle)</summary>

Start from the four counts - everything else is arithmetic on them. What are precision and recall when a class never occurs?
</details>

<details><summary>💡 Hint 2</summary>

`tp = np.sum((y_true == 1) & (y_pred == 1))` and so on. Guard each division: `x / y if y else 0.0`. Validate with `validate_labels(y_true)` and `validate_labels(y_pred, len(y_true))`.
</details>

In [ ]:
def binary_metrics(y_true, y_pred) -> dict:
    """accuracy, precision, recall, f1 (floats; 0.0 for a zero denominator) and tp, fp, tn, fn (ints) for labels -1 / +1."""
    # TODO: validate, count tp / fp / tn / fn, compute the four ratios safely.
    raise NotImplementedError("TODO: implement binary_metrics()")

In [ ]:
with if_ready():
    print(binary_metrics([1, 1, 1, 1, -1, -1], [1, 1, -1, -1, 1, -1]), "(hand check: tp 2, fp 1, tn 1, fn 2: accuracy 1/2, precision 2/3, recall 1/2, F1 4/7)")
    print("nothing predicted positive:", binary_metrics([1, -1, -1], [-1, -1, -1]))

In [ ]:
report = run_public_checks(8, globals(), keyword="binary_metrics")

## Part 7 · The whole experiment, without leakage  *(TODO)*
Complete `run_experiment(X_train, y_train, X_test, y_test, num_layers=2, seed=0, maxiter=100)` on **raw** data (running it takes about 20-30 seconds - every loss evaluation simulates the circuit once per training sample): scale (statistics from the training data), train on the training data only, predict with `+1 if f ≥ 0 else −1` and report the documented dictionary. The test data may be used for **evaluation only**.

<details><summary>💡 Hint 1 (subtle)</summary>

Which of your functions may see the test data, and which must not? If you change *only* the test set, which returned numbers must stay exactly the same?
</details>

<details><summary>💡 Hint 2</summary>

Call `scale_features(X_train, X_test)`, give only the scaled *training* data to `train_vqc`, predict labels on both scaled sets with the trained `params`, and compute `binary_metrics` for each. `feature_min` / `feature_max` are the minima / maxima of the **raw training** data.
</details>

In [ ]:
def run_experiment(X_train, y_train, X_test, y_test, num_layers: int = 2, seed: int = 0, maxiter: int = 100) -> dict:
    """Scale (train statistics), train, evaluate: params, num_layers, feature_min, feature_max, initial_loss, train_loss, test_loss, train_metrics, test_metrics."""
    # TODO: scale -> train_vqc on the training data only -> predict -> binary_metrics for train and test.
    raise NotImplementedError("TODO: implement run_experiment()")

In [ ]:
result = None                                           # filled in below once run_experiment works
with if_ready():
    result = run_experiment(X_train, y_train, X_test, y_test)
    print(f"loss: start {result['initial_loss']:.3f} -> train {result['train_loss']:.3f}, test {result['test_loss']:.3f}")
    print("train accuracy:", round(result["train_metrics"]["accuracy"], 3), "| test accuracy:", round(result["test_metrics"]["accuracy"], 3))
    print("test metrics:", {k: round(v, 3) for k, v in result["test_metrics"].items() if k in ("precision", "recall", "f1")})
    plot_decision_regions(lambda s: model_output(s, result["params"], result["num_layers"]), result["feature_min"], result["feature_max"],
                          X_train, y_train, X_test, y_test)

In [ ]:
report = run_public_checks(8, globals(), keyword="run_experiment")

## Part 8 · Is it any good?  *(not graded)*
Compare with two classical models that see the **same** split. With 60 test points, accuracies are uncertain by about ±10 points (the bars show 95 % intervals) - do not over-interpret small differences.

In [ ]:
if result is None:
    print("Finish Part 7 first: the comparison needs the result of your run_experiment.")
else:
    quantum = result["test_metrics"]["accuracy"]
    knn = float((knn_baseline(X_train, y_train, X_test) == y_test).mean())
    logistic = float((logistic_baseline(X_train, y_train, X_test) == y_test).mean())
    print(f"test accuracy: 2-qubit circuit {quantum:.3f} | logistic regression {logistic:.3f} | 5-nearest-neighbours {knn:.3f}")
    plot_accuracy_comparison({"2-qubit circuit": quantum, "logistic regression": logistic, "5-NN": knn}, len(y_test))

## Bonus · The exact gradient from circuit evaluations  *(optional)*
Gates of the form exp(−iθP/2) obey the **parameter-shift rule**: ∂f/∂θ = ½ [ f(θ + π/2) − f(θ − π/2) ]. Complete `parameter_shift_gradient(X, y, params, num_layers)`: the `4L` partial derivatives of the **loss** (chain rule: ∂L/∂θ = mean of 2 (f − y) ∂f/∂θ). It uses only circuit evaluations - the same ones a quantum computer can do, unlike finite differences of tiny steps.

<details><summary>💡 Hint</summary>

For parameter `k`: evaluate all outputs at `params + (π/2) e_k` and `params − (π/2) e_k`, take half the difference, multiply by `2 (f − y)` and average over the samples.
</details>

In [ ]:
def parameter_shift_gradient(X, y, params, num_layers: int) -> np.ndarray:
    """Gradient of the mean squared error with respect to all 4 * num_layers parameters (parameter-shift rule)."""
    # TODO (bonus): shifted evaluations of model_output, chain rule for the squared error.
    raise NotImplementedError("TODO (bonus): implement parameter_shift_gradient()")

In [ ]:
with if_ready():
    scaled_train, scaled_test = scale_features(X_train, X_test)
    sample_X, sample_y = scaled_train[:12], y_train[:12]
    shift_grad = parameter_shift_gradient(sample_X, sample_y, theta0, 2)
    eps = 1e-6
    finite = np.array([(mse_loss(sample_X, sample_y, theta0 + eps * e, 2) - mse_loss(sample_X, sample_y, theta0 - eps * e, 2)) / (2 * eps) for e in np.eye(8)])
    print("largest difference to central finite differences:", float(np.abs(shift_grad - finite).max()))

In [ ]:
report = run_public_checks(8, globals(), keyword="bonus")

## Reflection
1. The scaled test data may leave [0, π]. Why is that *correct*, and what would clipping or refitting hide?
2. The training accuracy is higher than the test accuracy. How big may that gap be before you worry, and what would you change (data, layers, iterations) - without peeking at the test set?
3. With 60 test points an accuracy of 0.85 has a 95 % interval of roughly 0.74 - 0.92. What would you need before claiming that the quantum model beats logistic regression?
4. Why can a nearest-neighbour model beat a two-qubit circuit with one encoding layer on this data? (Hint: what functions of the features can the circuit's output represent?)
5. (Bonus) Why is the parameter-shift rule attractive on real hardware, where only noisy expectation values are available?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 8)